# Level 1 · Gate 3 — AI judge for question → reply candidates (Colab free T4)

**Goal:** read every candidate pair (customer message + business reply) built locally by
`src/data/04_build_candidates.py` and decide, with a 7B open model, which ones are **real customer
enquiries** (`keep_question`) and which also have a **good reply** (`keep_pair`). Only kept data moves on
to memory building and fine-tuning — no garbage reaches training.

## What this notebook is — and is not

- **IS:** read-only scoring. The judge model only reads and answers in JSON. Nothing is trained here.
- **IS:** resumable. Every shard (1,000 pairs on a T4) is pushed to the private dataset repo
  `ankit-iiitdmj/mbg-judge-<export>` before the next one starts. A new session continues where the last stopped.
- **IS NOT:** fine-tuning. That is Level 3, a separate notebook.

## First-shard health check

After the FIRST shard of each session, `health_check()` aborts if more than 5% of judge answers are not
valid JSON, or if the judge says (almost) all yes / all no — so a broken prompt or model never burns hours of
free-T4 time.

## Pass criteria (Cell 11)

`keep_question` ≥ 10,000 pairs → **READY** for memory building (Level 1 step 6).
`keep_pair` is a bonus (good original replies); training answers are written from the business memory anyway.

In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL  = "https://github.com/ankit8453/chat-AI-model.git"   # private repo -> asks for a GitHub token once
REPO_PATH = "/content/chat-AI-model"

import os, subprocess, sys, getpass

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", url, REPO_PATH], capture_output=True, text=True)

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "README.md")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    r = _clone(REPO_URL)
    if r.returncode != 0:
        print("Public clone failed (private repo?). Paste a GitHub token with read access:")
        tok = getpass.getpass("GitHub token: ")
        r = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        if r.returncode != 0:
            raise SystemExit("git clone failed:\n" + r.stderr[-1500:])
        subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", REPO_URL], check=False)

os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
print("Repo root contents:", sorted(os.listdir(".")))

_pip_packages = ["huggingface_hub>=0.24", "transformers>=4.45", "accelerate>=0.33", "bitsandbytes>=0.43"]
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pip_packages], capture_output=True, text=True)
if proc.returncode != 0:
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tail above.")

# vLLM is optional: much faster, but newer releases may not support the T4. If it fails we fall back.
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
print("vllm install:", "ok" if proc.returncode == 0 else "FAILED (will use transformers 4-bit fallback)")
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
HF_NS = _whoami["name"]
print(f"HF Hub ok: user={HF_NS}")

In [ ]:
# Cell 4 — GPU check + auto config (model, shard size, batch)
import subprocess
subprocess.run(["nvidia-smi"], check=False)
print()

from judge.engine import auto_config
cfg = auto_config()
for k, v in cfg.items():
    print(f"  {k}: {v}")

## Judge configuration

- Export: **`messages_2026-10-31`** (change `EXPORT` in Cell 6 for later exports)
- Input: `ankit-iiitdmj/mbg-chat-candidates` → `<EXPORT>/candidates.jsonl` (PII-redacted, pushed by `05_push_candidates.py`)
- Model (T4): **Qwen2.5-7B-Instruct-AWQ** via vLLM, fallback **Qwen2.5-7B-Instruct** in 4-bit via transformers
- Greedy decoding (`temperature=0`), max 120 new tokens, one JSON verdict per pair
- Order: **AI-answered first**, then bot, agent, staff — the most valuable pairs are judged first
- Output: `ankit-iiitdmj/mbg-judge-<EXPORT>` → `shards/shard_XXXXXX.jsonl`, then `judged_all.jsonl` + `summary.json`

In [ ]:
# Cell 6 — load candidates from the private HF dataset
import json
from huggingface_hub import hf_hub_download

EXPORT = "messages_2026-10-31"
_p = hf_hub_download(f"{HF_NS}/mbg-chat-candidates", f"{EXPORT}/candidates.jsonl", repo_type="dataset")
with open(_p, encoding="utf-8") as fh:
    candidates = [json.loads(l) for l in fh if l.strip()]

PRIORITY = {"ai": 0, "bot": 1, "agent": 2, "staff": 3}
candidates.sort(key=lambda c: (PRIORITY.get(c["author"], 9), c["pair_id"]))

from collections import Counter
print(f"candidates: {len(candidates):,}")
print("by author:", dict(Counter(c["author"] for c in candidates)))
print("by language hint:", dict(Counter(c["lang_hint"] for c in candidates)))

In [ ]:
# Cell 7 — resume state from HF Hub (shards already judged in earlier sessions)
from judge.hub import JudgeShardManager

ckpt_mgr = JudgeShardManager(HF_NS, EXPORT)
done_rows, next_shard = ckpt_mgr.try_load_done()
done_ids = {r["pair_id"] for r in done_rows}
remaining = [c for c in candidates if c["pair_id"] not in done_ids]

print(f"repo: {ckpt_mgr.repo_id} (private)")
print(f"already judged: {len(done_rows):,} in {next_shard} shards")
print(f"remaining:      {len(remaining):,}")
if not remaining:
    print("Nothing left to judge — go to Cell 11.")

In [ ]:
# Cell 8 — build the judge engine (vLLM, else transformers 4-bit)
from judge.engine import build_engine
engine = build_engine(cfg, backend="auto")

In [ ]:
# Cell 9 — judging loop: one shard at a time, pushed to HF before the next; first-shard health check
import time, datetime
from judge.prompt import build_messages, parse_verdict, keep_decisions
from judge.hub import health_check

SHARD = cfg["shard_size"]
MAX_SHARDS_THIS_SESSION = None   # e.g. 3 to stop early on purpose; None = run until done / disconnected

t_start = time.time(); judged_now = 0; shard_idx = next_shard
for s, i in enumerate(range(0, len(remaining), SHARD)):
    if MAX_SHARDS_THIS_SESSION is not None and s >= MAX_SHARDS_THIS_SESSION:
        print("Reached MAX_SHARDS_THIS_SESSION — stopping cleanly."); break
    batch = remaining[i:i + SHARD]
    t0 = time.time()
    outputs = engine.generate([build_messages(c) for c in batch])
    rows = []
    for c, text in zip(batch, outputs):
        v = parse_verdict(text)
        rows.append({
            "pair_id": c["pair_id"], "business_id": c["business_id"], "author": c["author"],
            "lang_hint": c["lang_hint"], "verdict": v, **keep_decisions(v),
            "raw": None if v else (text or "")[:300],
            "engine": engine.name, "model": engine.model_id,
            "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        })
    dest = ckpt_mgr.save_shard(shard_idx, rows)
    if s == 0:
        health_check(rows)
    judged_now += len(rows); shard_idx += 1
    rate = judged_now / (time.time() - t_start)
    left = len(remaining) - (i + len(batch))
    kq = sum(r["keep_question"] for r in rows) / len(rows); kp = sum(r["keep_pair"] for r in rows) / len(rows)
    print(f"[shard {shard_idx-1}] {len(rows)} pairs in {time.time()-t0:.0f}s | keep_q={kq:.0%} keep_pair={kp:.0%} | "
          f"{rate:.1f} pairs/s | left {left:,} (~{left/max(rate,1e-6)/3600:.1f} h) | pushed {dest}")

print(f"\nThis session judged {judged_now:,} pairs.")

## Summary + verdict

Re-loads **all** shards from the Hub (so it works in any session), writes `judged_all.jsonl` and
`summary.json` to `ankit-iiitdmj/mbg-judge-<EXPORT>`, prints examples of kept and dropped pairs for a quick
human look, and gives the verdict.

In [ ]:
# Cell 11 — aggregate all shards, push judged_all.jsonl + summary.json, show examples, verdict
import json, random, tempfile, os
from collections import Counter

all_rows, _ = ckpt_mgr.try_load_done()
by_id = {c["pair_id"]: c for c in candidates}
n = len(all_rows)
kq = [r for r in all_rows if r["keep_question"]]
kp = [r for r in all_rows if r["keep_pair"]]

def share(rows, key):
    return dict(Counter(key(r) for r in rows).most_common())

summary = {
    "export": EXPORT, "judged": n, "candidates_total": len(candidates),
    "parse_failures": sum(1 for r in all_rows if r["verdict"] is None),
    "keep_question": len(kq), "keep_pair": len(kp),
    "keep_question_by_author": share(kq, lambda r: r["author"]),
    "keep_question_by_topic": share(kq, lambda r: r["verdict"]["topic"]),
    "keep_question_by_language": share(kq, lambda r: r["verdict"]["language"]),
    "businesses_with_kept_questions": len({r["business_id"] for r in kq}),
}
tmp = tempfile.mkdtemp()
with open(os.path.join(tmp, "judged_all.jsonl"), "w", encoding="utf-8") as fh:
    for r in all_rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
with open(os.path.join(tmp, "summary.json"), "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
ckpt_mgr.save_file(os.path.join(tmp, "judged_all.jsonl"), "judged_all.jsonl", "all judged pairs")
ckpt_mgr.save_file(os.path.join(tmp, "summary.json"), "summary.json", "judge summary")
print(json.dumps(summary, indent=2, ensure_ascii=False))

random.seed(0)
def show(rows, title, k=8):
    print(f"\n=== {title} ===")
    for r in random.sample(rows, min(k, len(rows))):
        c = by_id[r["pair_id"]]
        print(f"[{r['author']}] Q: {c['question'][:140]!r}\n        A: {c['reply'][:200]!r}\n        -> {r['verdict']}")
show(kp, "KEPT (question + reply)")
show([r for r in all_rows if r["keep_question"] and not r["keep_pair"]], "KEPT question only (reply not good enough)")
show([r for r in all_rows if not r["keep_question"]], "DROPPED")

done = n >= len(candidates)
print("\nProgress:", f"{n:,}/{len(candidates):,} judged", "(complete)" if done else "(run Cell 9 again next session)")
print("VERDICT:", "READY for memory building" if len(kq) >= 10_000 else
      ("NOT ENOUGH YET — keep judging" if not done else "NOT ENOUGH — wait for more exports"))

## Decision tree

- **READY for memory building** — `keep_question` ≥ 10,000. Locally: download `judged_all.jsonl` and run
  Level 1 step 6 (build each business's memory in our format). Spot-check the KEPT examples printed above
  first (Gate 4: Ankit reviews ~200).
- **NOT ENOUGH YET — keep judging** — free-T4 session ended before all shards were done. Open the notebook
  again tomorrow and Run all: Cell 7 resumes from the Hub.
- **NOT ENOUGH — wait for more exports** — everything judged but under 10k real enquiries. Push the next
  Chatrace export through steps 01–05 and judge it too (new `EXPORT` value).
- **ABORT in health check** — the judge's answers are broken (JSON or all-yes/all-no). Fix
  `src/judge/prompt.py` or switch model in `src/judge/engine.py`, push to GitHub, Run all again.